# Vaani KWS — V2 Dataset Preparation
---
**Project:** SIH 2026 — Low Latency & Efficient Voice Activator for Edge Devices  
**Keyword:** "Vaani"  
**Version:** V2  

## V2 Motivation
V1 achieved ~97.71% accuracy at threshold 0.90, but suffered catastrophic silence false-fires:
- 23/30 false activations on a ~6.93s quiet-room recording
- Average P(Vaani) = 0.871 on silence  
- Max P(Vaani) = 0.9985 on silence  

**V2 Goal:** Explicitly include silence/room-tone data and preserve negative diversity.

### What This Notebook Does
1. Discovers and validates all source audio
2. Performs stratified sampling of Speech Commands (~100/category)
3. Splits into train / validation / test (70 / 15 / 15)
4. Per-speaker 70/15/15 for positive data (every speaker in every split)
5. Source-aware splitting for silence and background (no cross-split leakage)
6. Copies selected files into `model_v2/data/`
7. Builds a reproducible manifest with SHA-256 hashes
8. Runs comprehensive leakage verification
9. Prints final dataset report

### What This Notebook Does NOT Do
- No augmentation (handled in notebook 02)
- No feature extraction / MFCC / log-Mel (handled in notebook 02)
- No neural-network training
- No TensorFlow imports
- No modification of original data under `dataset/raw/`

## 1. Configuration

In [ ]:
# ============================================================
# SECTION 1 - CONFIGURATION
# ============================================================

import os
import sys
import json
import wave
import shutil
import hashlib
import re
import math
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Audio requirements
EXPECTED_SR = 16000
EXPECTED_CHANNELS = 1
EXPECTED_SAMPLE_WIDTH = 2
EXPECTED_DURATION_S = 1.0
DURATION_TOLERANCE_S = 0.05

# Split ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Speech Commands sampling
TARGET_PER_SC_CATEGORY = 100
EXPECTED_SC_CATEGORIES = 35

# Required final selected counts
EXPECTED_COUNTS = {
    "positive": 704,
    "negative_silence": 307,
    "negative_background": 791,
    "negative_speech_commands": 3500,
    "grand_total": 5302,
}

# Labels
LABEL_POSITIVE = 1
LABEL_NEGATIVE = 0

SPLITS = ["train", "validation", "test"]
CATEGORIES = [
    "positive",
    "negative_silence",
    "negative_background",
    "negative_speech_commands",
]
SPLIT_RATIOS = {
    "train": TRAIN_RATIO,
    "validation": VAL_RATIO,
    "test": TEST_RATIO,
}


def target_counts_for_total(total):
    """Return integer split targets using largest-remainder rounding."""
    exact = {split: total * SPLIT_RATIOS[split] for split in SPLITS}
    targets = {split: int(math.floor(exact[split])) for split in SPLITS}
    remainder = total - sum(targets.values())

    order = sorted(
        SPLITS,
        key=lambda split: (-(exact[split] - targets[split]), SPLITS.index(split)),
    )
    for split in order[:remainder]:
        targets[split] += 1

    if total >= len(SPLITS):
        for split in SPLITS:
            if targets[split] == 0:
                donor = max(SPLITS, key=lambda s: targets[s])
                targets[donor] -= 1
                targets[split] += 1

    return targets


def assign_groups_to_splits(groups, rng, require_all_splits=True):
    """Assign indivisible file groups to train/validation/test near 70/15/15."""
    prepared = []
    for group in groups:
        files = sorted(list(group["files"]), key=lambda p: str(p).lower())
        if not files:
            continue
        prepared.append({
            **group,
            "files": files,
            "size": len(files),
        })

    total = sum(group["size"] for group in prepared)
    targets = target_counts_for_total(total)
    assignments = {split: [] for split in SPLITS}
    counts = Counter()

    if not prepared:
        return assignments, targets, {split: 0 for split in SPLITS}

    order = list(range(len(prepared)))
    rng.shuffle(order)
    pending = [prepared[i] for i in order]

    if require_all_splits and len(pending) >= len(SPLITS):
        for split in sorted(SPLITS, key=lambda s: (targets[s], SPLITS.index(s))):
            idx = min(
                range(len(pending)),
                key=lambda i: (pending[i]["size"], str(pending[i]["files"][0]).lower()),
            )
            group = pending.pop(idx)
            assignments[split].append(group)
            counts[split] += group["size"]

    for group in pending:
        def score(split):
            next_counts = {s: counts[s] for s in SPLITS}
            next_counts[split] += group["size"]
            total_error = sum(abs(next_counts[s] - targets[s]) for s in SPLITS)
            deficit_before = targets[split] - counts[split]
            relative_deficit = deficit_before / max(targets[split], 1)
            overshoot = max(0, next_counts[split] - targets[split])
            return (total_error, -relative_deficit, overshoot, counts[split], SPLITS.index(split))

        chosen = min(SPLITS, key=score)
        assignments[chosen].append(group)
        counts[chosen] += group["size"]

    return assignments, targets, {split: counts[split] for split in SPLITS}


def file_sha256(filepath):
    """Compute SHA-256 hash of a file."""
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def get_wav_info(filepath):
    """Get sample_rate, channels, duration from a WAV file."""
    try:
        with wave.open(str(filepath), "rb") as wf:
            sr = wf.getframerate()
            ch = wf.getnchannels()
            dur = wf.getnframes() / sr
            return sr, ch, dur
    except Exception:
        return None, None, None


print("Configuration loaded.")
print(f"  SEED = {SEED}")
print(f"  Split ratios: train={TRAIN_RATIO}, val={VAL_RATIO}, test={TEST_RATIO}")
print(f"  Target Speech Commands per category: {TARGET_PER_SC_CATEGORY}")
print(f"  Expected final counts: {EXPECTED_COUNTS}")

## 2. Locate Project Root

In [ ]:
# ============================================================
# SECTION 2 — PROJECT ROOT
# ============================================================

def find_project_root():
    """Walk up from CWD to find the project root
    (directory containing both 'dataset/' and 'model_v1/').
    """
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "dataset").is_dir() and (candidate / "model_v1").is_dir():
            return candidate
    # Hardcoded fallback
    fallback = Path(r"c:/Users/Mayank Singh/Codes/SIH 2026")
    if fallback.is_dir():
        return fallback
    raise FileNotFoundError(
        "Could not find PROJECT_ROOT. "
        "Expected a directory containing 'dataset/' and 'model_v1/'."
    )

PROJECT_ROOT = find_project_root()
DATASET_RAW  = PROJECT_ROOT / "dataset" / "raw"
MODEL_V2     = PROJECT_ROOT / "model_v2"
TEMP_DIR     = PROJECT_ROOT / "temp"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATASET_RAW:  {DATASET_RAW}")
print(f"MODEL_V2:     {MODEL_V2}")

# ── Source directories ───────────────────────────────────────
POSITIVE_DIR = DATASET_RAW / "positive"
assert POSITIVE_DIR.is_dir(), f"Missing: {POSITIVE_DIR}"

# Silence — check multiple possible locations
_silence_candidates = [
    DATASET_RAW / "negative_silence",
    DATASET_RAW / "silence" / "processed",
    DATASET_RAW / "silence",
]
SILENCE_DIR = None
for _sc in _silence_candidates:
    if _sc.is_dir() and list(_sc.glob("*.wav")):
        SILENCE_DIR = _sc
        break
if SILENCE_DIR is None:
    raise FileNotFoundError(
        f"Could not find silence WAV directory. Checked: {_silence_candidates}"
    )

BACKGROUND_DIR = DATASET_RAW / "negative_background"
assert BACKGROUND_DIR.is_dir(), f"Missing: {BACKGROUND_DIR}"

SPEECH_CMD_DIR = DATASET_RAW / "negative_speech_commands"
assert SPEECH_CMD_DIR.is_dir(), f"Missing: {SPEECH_CMD_DIR}"

print()
print(f"Positive dir:        {POSITIVE_DIR}")
print(f"Silence dir:         {SILENCE_DIR}")
print(f"Background dir:      {BACKGROUND_DIR}")
print(f"Speech commands dir: {SPEECH_CMD_DIR}")

In [ ]:
# ── Create V2 directory structure ─────────────────────────────

V2_DIRS = {
    "data":       MODEL_V2 / "data",
    "features":   MODEL_V2 / "features",
    "checkpoints": MODEL_V2 / "checkpoints",
    "evaluation": MODEL_V2 / "evaluation",
    "exports":    MODEL_V2 / "exports",
    "scripts":    MODEL_V2 / "scripts",
    "manifests":  MODEL_V2 / "data" / "manifests",
}

for name, path in V2_DIRS.items():
    path.mkdir(parents=True, exist_ok=True)

for split in SPLITS:
    for cat in CATEGORIES:
        (V2_DIRS["data"] / split / cat).mkdir(parents=True, exist_ok=True)

# Temp directory for any intermediate files
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print("V2 directory structure created.")

## 3. Source Dataset Inspection

In [ ]:
# ============================================================
# SECTION 3 - DISCOVER SOURCE FILES
# ============================================================

def sorted_wavs(root):
    """Discover WAV files recursively with a stable order."""
    return sorted(
        [p for p in root.rglob("*.wav") if p.is_file()],
        key=lambda p: str(p.relative_to(root)).replace("\\", "/").lower(),
    )


def require_count(label, actual, expected):
    if actual != expected:
        raise RuntimeError(
            f"{label} count mismatch: discovered {actual}, expected {expected}. "
            "Do not continue to splitting/training."
        )


positive_files = sorted_wavs(POSITIVE_DIR)
silence_files = sorted_wavs(SILENCE_DIR)
background_files = sorted_wavs(BACKGROUND_DIR)
speech_cmd_files = sorted_wavs(SPEECH_CMD_DIR)

require_count("positive WAV files under dataset/raw/positive", len(positive_files), EXPECTED_COUNTS["positive"])
require_count("silence WAV files", len(silence_files), EXPECTED_COUNTS["negative_silence"])
require_count("background WAV files", len(background_files), EXPECTED_COUNTS["negative_background"])

positive_rel_paths = [str(p.relative_to(POSITIVE_DIR)).replace("\\", "/") for p in positive_files]
if len(positive_rel_paths) != len(set(positive_rel_paths)):
    raise RuntimeError("Duplicate positive relative paths discovered; cannot build a unique manifest.")

positive_filename_counts = Counter(p.name for p in positive_files)
duplicate_positive_filenames = sorted(name for name, count in positive_filename_counts.items() if count > 1)
if duplicate_positive_filenames:
    raise RuntimeError(
        "Positive filenames must be unique because copied split folders are flat. "
        f"Duplicate filenames: {duplicate_positive_filenames[:10]}"
    )

print("=" * 60)
print("SOURCE FILE COUNTS")
print("=" * 60)
print(f"  Positive (Vaani):         {len(positive_files):>7,}")
print(f"  Negative silence:         {len(silence_files):>7,}")
print(f"  Negative background:      {len(background_files):>7,}")
print(f"  Negative speech commands: {len(speech_cmd_files):>7,}")
print(f"  {'-' * 44}")
total_source = (
    len(positive_files)
    + len(silence_files)
    + len(background_files)
    + len(speech_cmd_files)
)
print(f"  Total source files:       {total_source:>7,}")
print()
print(f"Verified positive discovery from {POSITIVE_DIR}")
print(f"  Positive WAV files discovered: {len(positive_files)}")
print(f"  Unique positive relative paths: {len(set(positive_rel_paths))}")
print(f"  Unique positive filenames:      {len(set(positive_filename_counts))}")


def extract_positive_speaker(filepath):
    """Positive filenames: <speaker>_<number>.wav -> speaker name."""
    parts = filepath.stem.split("_")
    if len(parts) > 1 and parts[-1].isdigit():
        return "_".join(parts[:-1])
    return "unknown"


pos_speakers = defaultdict(list)
for f in positive_files:
    spk = extract_positive_speaker(f)
    pos_speakers[spk].append(f)

print()
print("Positive speakers:")
for spk in sorted(pos_speakers):
    print(f"  {spk:<12s}: {len(pos_speakers[spk]):>4d} recordings")


def extract_silence_source(filepath):
    """Silence source is everything before '_clip_'."""
    m = re.match(r"^(.+?)_clip_", filepath.name)
    if m:
        return m.group(1)
    return filepath.stem


silence_sources = defaultdict(list)
for f in silence_files:
    src = extract_silence_source(f)
    silence_sources[src].append(f)

print()
print("Silence source recordings:")
for src in sorted(silence_sources):
    print(f"  {src!r}: {len(silence_sources[src]):>4d} clips")


def extract_background_source(filepath):
    """Background source is filename without trailing numeric suffix."""
    name = filepath.stem
    m = re.match(r"^(.+)_\d{4,}$", name)
    if m:
        return m.group(1)
    return name


bg_sources = defaultdict(list)
for f in background_files:
    src = extract_background_source(f)
    bg_sources[src].append(f)

print()
print("Background source recordings:")
for src in sorted(bg_sources):
    print(f"  {src}: {len(bg_sources[src]):>4d} clips")

## 4. Audio Validation

Validate that all source files meet requirements:
- Decodable WAV
- 16 kHz sample rate
- Mono (1 channel)
- PCM16 (2-byte samples)
- ~1 second duration (±50ms tolerance)

Files that fail are **reported but not silently deleted**.

In [ ]:
# ============================================================
# SECTION 4 - AUDIO VALIDATION
# ============================================================

def validate_wav(filepath):
    """Validate a single WAV file. Returns dict with properties and issues."""
    result = {
        "filepath": str(filepath),
        "filename": filepath.name,
        "readable": False,
        "sample_rate": None,
        "channels": None,
        "sample_width": None,
        "duration_s": None,
        "issues": [],
    }

    try:
        with wave.open(str(filepath), "rb") as wf:
            result["sample_rate"] = wf.getframerate()
            result["channels"] = wf.getnchannels()
            result["sample_width"] = wf.getsampwidth()
            result["duration_s"] = wf.getnframes() / wf.getframerate()
            result["readable"] = True
    except Exception as e:
        result["issues"].append(f"Cannot read WAV header: {e}")
        return result

    if result["sample_rate"] != EXPECTED_SR:
        result["issues"].append(f"SR={result['sample_rate']} (expected {EXPECTED_SR})")
    if result["channels"] != EXPECTED_CHANNELS:
        result["issues"].append(f"CH={result['channels']} (expected {EXPECTED_CHANNELS})")
    if result["sample_width"] != EXPECTED_SAMPLE_WIDTH:
        result["issues"].append(f"SW={result['sample_width']} (expected {EXPECTED_SAMPLE_WIDTH})")
    if result["duration_s"] is not None:
        if abs(result["duration_s"] - EXPECTED_DURATION_S) > DURATION_TOLERANCE_S:
            result["issues"].append(
                f"Duration={result['duration_s']:.3f}s (expected {EXPECTED_DURATION_S}s)"
            )
    return result


def validate_file_set(files, category_name):
    """Validate every WAV file in a selected file set."""
    results = []
    for f in tqdm(files, desc=f"Validating {category_name}", leave=True):
        r = validate_wav(f)
        r["category"] = category_name
        results.append(r)

    issues_count = sum(1 for r in results if r["issues"])
    unreadable = sum(1 for r in results if not r["readable"])
    print(f"  Total: {len(files)}, Issues: {issues_count}, Unreadable: {unreadable}")

    if issues_count > 0:
        print("  Files with issues:")
        for r in results:
            if r["issues"]:
                print(f"    {r['filename']}: {'; '.join(r['issues'])}")

    return results


print("=" * 60)
print("VALIDATING ALL NON-SPEECH-COMMAND SOURCE FILES")
print("=" * 60)
print()

val_positive = validate_file_set(positive_files, "positive")
print()
val_silence = validate_file_set(silence_files, "negative_silence")
print()
val_background = validate_file_set(background_files, "negative_background")
print()
print("Selected Speech Commands will be validated after stratified selection.")

all_val_results = val_positive + val_silence + val_background
total_issues = sum(1 for r in all_val_results if r["issues"])
total_unreadable = sum(1 for r in all_val_results if not r["readable"])

print()
print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
print(f"  Total validated so far: {len(all_val_results)}")
print(f"  Files with issues:     {total_issues}")
print(f"  Unreadable files:      {total_unreadable}")

if total_issues:
    raise RuntimeError("Audio validation failed for source files. Do not continue.")

print("  All validated files pass checks so far.")

## 5. Speech Commands — Category Discovery

In [ ]:
# ============================================================
# SECTION 5 — SPEECH COMMANDS CATEGORY DISCOVERY
# ============================================================

def extract_sc_category(filepath):
    """Extract word/category from Speech Commands filename.
    Format: <word>_<speakerhash>_nohash_<idx>.wav
    Category is everything before the first 8-char hex hash.
    """
    parts = filepath.stem.split('_')
    category_parts = []
    for p in parts:
        if len(p) >= 8 and all(c in '0123456789abcdef' for c in p):
            break
        category_parts.append(p)
    return '_'.join(category_parts) if category_parts else parts[0]

def extract_sc_speaker(filepath):
    """Extract speaker hash from Speech Commands filename."""
    parts = filepath.stem.split('_')
    for p in parts:
        if len(p) == 8 and all(c in '0123456789abcdef' for c in p):
            return p
    return "unknown"

# Build category → files mapping
sc_categories = defaultdict(list)
for f in speech_cmd_files:
    cat = extract_sc_category(f)
    sc_categories[cat].append(f)
sc_categories = dict(sorted(sc_categories.items()))

print("=" * 60)
print("SPEECH COMMANDS — CATEGORIES")
print("=" * 60)
print(f"Total categories discovered: {len(sc_categories)}")
print(f"Total files: {len(speech_cmd_files):,}")
print()
print(f"{'Category':<20s} {'Count':>8s}")
print("─" * 30)
for cat, files in sc_categories.items():
    print(f"{cat:<20s} {len(files):>8,}")
print("─" * 30)
print(f"{'TOTAL':<20s} {sum(len(v) for v in sc_categories.values()):>8,}")

## 6. Speech Commands — Stratified Sampling (~100/category)

In [ ]:
# ============================================================
# SECTION 6 - SPEECH COMMANDS STRATIFIED SAMPLING
# ============================================================

if len(sc_categories) != EXPECTED_SC_CATEGORIES:
    raise RuntimeError(
        f"Speech Commands category mismatch: found {len(sc_categories)}, "
        f"expected {EXPECTED_SC_CATEGORIES}."
    )

rng_sc = np.random.RandomState(SEED)

selected_sc = {}
sc_selection_report = []

print("=" * 60)
print("SPEECH COMMANDS - STRATIFIED SAMPLING")
print("=" * 60)
print(f"Target per category: {TARGET_PER_SC_CATEGORY}")
print()
print(f"{'Category':<20s} {'Available':>10s} {'Selected':>10s} {'Note'}")
print("-" * 65)

total_selected = 0
for cat, files in sc_categories.items():
    n_available = len(files)
    n_select = min(TARGET_PER_SC_CATEGORY, n_available)

    if n_available < TARGET_PER_SC_CATEGORY:
        raise RuntimeError(
            f"Speech Commands category {cat!r} has only {n_available} files; "
            f"expected at least {TARGET_PER_SC_CATEGORY}."
        )

    indices = rng_sc.choice(n_available, size=n_select, replace=False)
    chosen = [files[i] for i in sorted(indices)]
    note = ""

    selected_sc[cat] = chosen
    total_selected += len(chosen)
    sc_selection_report.append({
        "category": cat,
        "available": n_available,
        "selected": len(chosen),
    })
    print(f"{cat:<20s} {n_available:>10,} {len(chosen):>10,} {note}")

print("-" * 65)
print(f"{'TOTAL':<20s} {sum(len(v) for v in sc_categories.values()):>10,} {total_selected:>10,}")

selected_sc_files = []
for cat, files in selected_sc.items():
    for f in files:
        selected_sc_files.append((f, cat))

if len(selected_sc_files) != EXPECTED_COUNTS["negative_speech_commands"]:
    raise RuntimeError(
        f"Selected Speech Commands mismatch: {len(selected_sc_files)} != "
        f"{EXPECTED_COUNTS['negative_speech_commands']}"
    )

print(f"\nTotal Speech Commands files selected: {len(selected_sc_files)}")
print("Validating every selected Speech Commands file...")
selected_sc_paths = [f for f, _ in selected_sc_files]
val_sc_selected = validate_file_set(selected_sc_paths, "negative_speech_commands")

all_val_results = val_positive + val_silence + val_background + val_sc_selected
total_issues = sum(1 for r in all_val_results if r["issues"])
total_unreadable = sum(1 for r in all_val_results if not r["readable"])

print()
print("=" * 60)
print("FULL SELECTED DATASET VALIDATION SUMMARY")
print("=" * 60)
print(f"  Total selected files validated: {len(all_val_results)}")
print(f"  Files with issues:             {total_issues}")
print(f"  Unreadable files:              {total_unreadable}")

if len(all_val_results) != EXPECTED_COUNTS["grand_total"]:
    raise RuntimeError(
        f"Validation coverage mismatch: validated {len(all_val_results)} files, "
        f"expected {EXPECTED_COUNTS['grand_total']}."
    )

if total_issues:
    raise RuntimeError("Audio validation failed for selected files. Do not continue.")

print("All selected files pass validation.")

# Save selection list for reproducibility.
sel_rows = []
for f, cat in selected_sc_files:
    sel_rows.append({
        "filepath": str(f),
        "filename": f.name,
        "category": cat,
        "speaker": extract_sc_speaker(f),
    })
sel_df = pd.DataFrame(sel_rows)
sel_path = V2_DIRS["manifests"] / "speech_commands_selection.csv"
sel_df.to_csv(sel_path, index=False)
print(f"Saved selection list: {sel_path}")

## 7. Positive Split - Per-Speaker, SHA-Aware 70/15/15

Every discovered positive WAV file must appear in the manifest exactly once.
Byte-identical positive WAVs are kept, but each identical-audio SHA-256 group
is assigned to one split only so identical audio cannot cross train,
validation, and test.

In [ ]:
# ============================================================
# SECTION 7 - POSITIVE SPLIT (PER-SPEAKER, SHA-AWARE)
# ============================================================

positive_records = []
positive_hash_by_file = {}

print("=" * 60)
print("POSITIVE SPLIT - PER-SPEAKER, SHA-AWARE")
print("=" * 60)
print()
print("Hashing all discovered positive WAV files before splitting...")
for f in tqdm(positive_files, desc="Hashing positive WAVs"):
    positive_hash_by_file[f] = file_sha256(f)

positive_hash_groups = defaultdict(list)
for f, h in positive_hash_by_file.items():
    positive_hash_groups[h].append(f)

duplicate_positive_hashes = {
    h: files for h, files in positive_hash_groups.items() if len(files) > 1
}
cross_speaker_hashes = []
for h, files in duplicate_positive_hashes.items():
    speakers = sorted({extract_positive_speaker(f) for f in files})
    if len(speakers) > 1:
        cross_speaker_hashes.append((h, speakers, files))

print(f"Positive WAV files discovered:       {len(positive_files)}")
print(f"Unique positive SHA-256 groups:      {len(positive_hash_groups)}")
print(f"Duplicate positive SHA-256 groups:   {len(duplicate_positive_hashes)}")
print("Duplicate positive audio is kept and assigned as indivisible split groups.")

if cross_speaker_hashes:
    print("Cross-speaker positive duplicate hashes found:")
    for h, speakers, files in cross_speaker_hashes[:10]:
        names = [f.name for f in files]
        print(f"  {h[:16]}... speakers={speakers} files={names}")
    raise RuntimeError(
        "Positive duplicate hashes span speakers. Resolve speaker/source identity "
        "before per-speaker splitting."
    )

print()
print(f"{'Speaker':<12s} {'Files':>6s} {'Groups':>6s} {'Train':>6s} {'Val':>6s} {'Test':>6s}  "
      f"{'Target train/val/test':>24s}")
print("-" * 86)

rng_pos = np.random.RandomState(SEED)

for speaker in sorted(pos_speakers.keys()):
    files = pos_speakers[speaker]
    hash_groups = defaultdict(list)
    for f in files:
        hash_groups[positive_hash_by_file[f]].append(f)

    groups = [
        {"sha256": h, "files": grouped_files}
        for h, grouped_files in sorted(
            hash_groups.items(),
            key=lambda item: str(sorted(item[1], key=lambda p: p.name)[0]).lower(),
        )
    ]

    assignments, targets, counts = assign_groups_to_splits(groups, rng_pos, require_all_splits=True)

    for split in SPLITS:
        for group in assignments[split]:
            for f in group["files"]:
                positive_records.append({
                    "filepath": f,
                    "speaker": speaker,
                    "split": split,
                    "sha256": group["sha256"],
                })

    print(
        f"{speaker:<12s} {len(files):>6d} {len(groups):>6d} "
        f"{counts.get('train', 0):>6d} {counts.get('validation', 0):>6d} "
        f"{counts.get('test', 0):>6d}  "
        f"{targets['train']:>6d}/{targets['validation']:>3d}/{targets['test']:<3d}"
    )

totals = Counter(r["split"] for r in positive_records)
print("-" * 86)
print(
    f"{'TOTAL':<12s} {len(positive_records):>6d} {'':>6s} "
    f"{totals['train']:>6d} {totals['validation']:>6d} {totals['test']:>6d}"
)
print()
print(f"positive train + validation + test == {len(positive_records)}")

if len(positive_records) != EXPECTED_COUNTS["positive"]:
    raise RuntimeError(
        f"Positive split lost files: {len(positive_records)} != {EXPECTED_COUNTS['positive']}"
    )

speaker_split_ok = True
for speaker in sorted(pos_speakers):
    speaker_splits = {r["split"] for r in positive_records if r["speaker"] == speaker}
    if speaker_splits != set(SPLITS):
        speaker_split_ok = False
        print(f"Speaker {speaker!r} appears only in {speaker_splits}")

if not speaker_split_ok:
    raise RuntimeError("Positive speaker representation failed.")

positive_hash_splits = defaultdict(set)
for rec in positive_records:
    positive_hash_splits[rec["sha256"]].add(rec["split"])

positive_cross_split_hashes = {
    h: splits for h, splits in positive_hash_splits.items() if len(splits) > 1
}
if positive_cross_split_hashes:
    raise RuntimeError(
        f"Positive identical audio crosses splits for {len(positive_cross_split_hashes)} hashes."
    )

print("Every positive speaker appears in train, validation, and test.")
print("No positive SHA-256 group crosses train/validation/test.")

print()
print("=" * 60)
print("EXACT POSITIVE FILENAMES BY SPLIT")
print("=" * 60)
for split in SPLITS:
    split_names = sorted(r["filepath"].name for r in positive_records if r["split"] == split)
    print(f"\n{split.upper()} positive files ({len(split_names)}):")
    for name in split_names:
        print(f"  {name}")

## 8. Silence & Background — Source-Aware Split

**Critical leakage prevention:** clips from the same original recording
must ALL go to the same split. We split by *source recording*, not by
individual clip.

### Silence
All silence clips come from a single source recording (`Recording (53)`).
Source-aware splitting means **all 307 clips must go to one split**.
We put them ALL in **train** since we cannot split a single source.
This is reported transparently.

### Background
6 distinct source recordings. We assign entire sources to splits.

In [ ]:
# ============================================================
# SECTION 8 — SILENCE SOURCE-AWARE SPLIT
# ============================================================

print("=" * 60)
print("SILENCE — SOURCE-AWARE SPLIT")
print("=" * 60)

silence_records = []
n_silence_sources = len(silence_sources)

print(f"\nSilence source recordings found: {n_silence_sources}")
for src, clips in sorted(silence_sources.items()):
    print(f"  {src!r}: {len(clips)} clips")

if n_silence_sources == 1:
    # ALL clips from one source → must all go to same split
    sole_source = list(silence_sources.keys())[0]
    print(f"\n⚠ LIMITATION: All {len(silence_files)} silence clips come from ONE source "
          f"recording ({sole_source!r}).")
    print("  Source-aware splitting requires all clips to stay together.")
    print("  → Assigning ALL silence clips to TRAIN.")
    print("  → Validation and test will have 0 silence clips.")
    print("  → This is reported transparently; a separate unseen-silence")
    print("    evaluation should be conducted when new recordings are available.")
    
    for f in silence_files:
        silence_records.append({
            "filepath": f,
            "source_recording_id": sole_source,
            "split": "train",
        })
else:
    # Multiple sources — split source-level
    source_names = sorted(silence_sources.keys())
    rng_sil = np.random.RandomState(SEED)
    rng_sil.shuffle(source_names)
    
    n_src = len(source_names)
    n_train_src = max(1, int(round(n_src * TRAIN_RATIO)))
    n_val_src   = max(1, int(round(n_src * VAL_RATIO)))
    n_test_src  = n_src - n_train_src - n_val_src
    if n_test_src < 1:
        n_test_src = 1
        n_train_src = n_src - n_val_src - n_test_src
    
    train_srcs = source_names[:n_train_src]
    val_srcs   = source_names[n_train_src:n_train_src + n_val_src]
    test_srcs  = source_names[n_train_src + n_val_src:]
    
    for src in train_srcs:
        for f in silence_sources[src]:
            silence_records.append({"filepath": f, "source_recording_id": src, "split": "train"})
    for src in val_srcs:
        for f in silence_sources[src]:
            silence_records.append({"filepath": f, "source_recording_id": src, "split": "validation"})
    for src in test_srcs:
        for f in silence_sources[src]:
            silence_records.append({"filepath": f, "source_recording_id": src, "split": "test"})

sil_split_counts = Counter(r["split"] for r in silence_records)
print(f"\nSilence split counts:")
for s in SPLITS:
    print(f"  {s:12s}: {sil_split_counts.get(s, 0)}")

In [ ]:
# ============================================================
# BACKGROUND — SOURCE-AWARE SPLIT
# ============================================================

print("=" * 60)
print("BACKGROUND — SOURCE-AWARE SPLIT")
print("=" * 60)

bg_records = []
n_bg_sources = len(bg_sources)
source_names_bg = sorted(bg_sources.keys())

print(f"\nBackground source recordings: {n_bg_sources}")
for src in source_names_bg:
    print(f"  {src}: {len(bg_sources[src])} clips")

if n_bg_sources < 3:
    print(f"\n⚠ Only {n_bg_sources} background sources — cannot split into 3 groups.")
    print("  Assigning all to TRAIN.")
    for src in source_names_bg:
        for f in bg_sources[src]:
            bg_records.append({"filepath": f, "source_recording_id": src, "split": "train"})
else:
    # Assign sources to splits proportionally
    rng_bg = np.random.RandomState(SEED)
    rng_bg.shuffle(source_names_bg)
    
    n_src = len(source_names_bg)
    n_train_src = max(1, int(round(n_src * TRAIN_RATIO)))
    n_val_src   = max(1, int(round(n_src * VAL_RATIO)))
    n_test_src  = n_src - n_train_src - n_val_src
    if n_test_src < 1:
        n_test_src = 1
        n_train_src = n_src - n_val_src - n_test_src
    
    train_bg = source_names_bg[:n_train_src]
    val_bg   = source_names_bg[n_train_src:n_train_src + n_val_src]
    test_bg  = source_names_bg[n_train_src + n_val_src:]
    
    print(f"\nSource assignment:")
    print(f"  Train sources ({len(train_bg)}): {train_bg}")
    print(f"  Val sources   ({len(val_bg)}):   {val_bg}")
    print(f"  Test sources  ({len(test_bg)}):  {test_bg}")
    
    for src in train_bg:
        for f in bg_sources[src]:
            bg_records.append({"filepath": f, "source_recording_id": src, "split": "train"})
    for src in val_bg:
        for f in bg_sources[src]:
            bg_records.append({"filepath": f, "source_recording_id": src, "split": "validation"})
    for src in test_bg:
        for f in bg_sources[src]:
            bg_records.append({"filepath": f, "source_recording_id": src, "split": "test"})

bg_split_counts = Counter(r["split"] for r in bg_records)
print(f"\nBackground split counts:")
for s in SPLITS:
    print(f"  {s:12s}: {bg_split_counts.get(s, 0)}")
total_bg = sum(bg_split_counts.values())
print(f"  {'TOTAL':12s}: {total_bg}")
for s in SPLITS:
    pct = bg_split_counts.get(s, 0) / total_bg * 100 if total_bg else 0
    print(f"  {s} ratio: {pct:.1f}%")
print()
print("Note: Ratios may differ from 70/15/15 because source integrity")
print("takes priority over exact percentages.")

In [ ]:
# ============================================================
# SPEECH COMMANDS - STRATIFIED, SHA-AWARE SPLIT
# ============================================================

sc_records = []
selected_sc_hash_by_file = {}

print("=" * 60)
print("SPEECH COMMANDS - SHA-AWARE SPLIT")
print("=" * 60)
print("Hashing selected Speech Commands files before splitting...")
for f, _ in tqdm(selected_sc_files, desc="Hashing selected Speech Commands"):
    selected_sc_hash_by_file[f] = file_sha256(f)

rng_sc_split = np.random.RandomState(SEED)

print()
print(f"{'Category':<20s} {'Total':>6s} {'Groups':>6s} {'Train':>6s} {'Val':>6s} {'Test':>6s}")
print("-" * 60)

for cat in sorted(selected_sc):
    files = selected_sc[cat]
    hash_groups = defaultdict(list)
    for f in files:
        hash_groups[selected_sc_hash_by_file[f]].append(f)

    groups = [
        {"sha256": h, "files": grouped_files}
        for h, grouped_files in sorted(
            hash_groups.items(),
            key=lambda item: str(sorted(item[1], key=lambda p: p.name)[0]).lower(),
        )
    ]
    assignments, targets, counts = assign_groups_to_splits(groups, rng_sc_split, require_all_splits=True)

    for split in SPLITS:
        for group in assignments[split]:
            for f in group["files"]:
                sc_records.append({
                    "filepath": f,
                    "subcategory": cat,
                    "split": split,
                    "sha256": group["sha256"],
                })

    print(
        f"{cat:<20s} {len(files):>6d} {len(groups):>6d} "
        f"{counts.get('train', 0):>6d} {counts.get('validation', 0):>6d} {counts.get('test', 0):>6d}"
    )


def co_locate_records_by_hash(records, label):
    """Move any remaining same-hash records to one split without dropping rows."""
    by_hash = defaultdict(list)
    for idx, rec in enumerate(records):
        by_hash[rec["sha256"]].append(idx)

    current_counts = Counter(rec["split"] for rec in records)
    moved_groups = 0
    moved_records = 0

    for h, indices in by_hash.items():
        splits = {records[i]["split"] for i in indices}
        if len(splits) <= 1:
            continue

        split_counts = Counter(records[i]["split"] for i in indices)
        target_split = sorted(
            SPLITS,
            key=lambda split: (
                -split_counts.get(split, 0),
                current_counts.get(split, 0),
                SPLITS.index(split),
            ),
        )[0]

        moved_groups += 1
        for i in indices:
            old_split = records[i]["split"]
            if old_split != target_split:
                records[i]["split"] = target_split
                current_counts[old_split] -= 1
                current_counts[target_split] += 1
                moved_records += 1

    print(f"\n{label}: co-located {moved_records} records across {moved_groups} duplicate hash groups.")


co_locate_records_by_hash(sc_records, "Speech Commands")

sc_split_counts = Counter(r["split"] for r in sc_records)
print()
print("Speech Commands split counts:")
for split in SPLITS:
    print(f"  {split:12s}: {sc_split_counts.get(split, 0)}")
print(f"  {'total':12s}: {sum(sc_split_counts.values())}")

if len(sc_records) != EXPECTED_COUNTS["negative_speech_commands"]:
    raise RuntimeError(
        f"Speech Commands split count mismatch: {len(sc_records)} != "
        f"{EXPECTED_COUNTS['negative_speech_commands']}"
    )

sc_hash_splits = defaultdict(set)
for rec in sc_records:
    sc_hash_splits[rec["sha256"]].add(rec["split"])
sc_cross_split_hashes = {h: splits for h, splits in sc_hash_splits.items() if len(splits) > 1}
if sc_cross_split_hashes:
    raise RuntimeError(
        f"Speech Commands identical audio crosses splits for {len(sc_cross_split_hashes)} hashes."
    )

sc_df_check = pd.DataFrame(sc_records)
all_sc_cats = set(sc_df_check["subcategory"].unique())
if len(all_sc_cats) != EXPECTED_SC_CATEGORIES:
    raise RuntimeError(
        f"Speech Commands category mismatch after split: {len(all_sc_cats)} != {EXPECTED_SC_CATEGORIES}"
    )

category_balance_ok = True
for cat in sorted(all_sc_cats):
    cat_total = len(sc_df_check[sc_df_check["subcategory"] == cat])
    if cat_total != TARGET_PER_SC_CATEGORY:
        category_balance_ok = False
        print(f"  Category {cat!r} has {cat_total} files (expected {TARGET_PER_SC_CATEGORY})")

for split in SPLITS:
    split_cats = set(sc_df_check[sc_df_check["split"] == split]["subcategory"])
    missing = all_sc_cats - split_cats
    if missing:
        category_balance_ok = False
        print(f"  {split} missing Speech Commands categories: {sorted(missing)}")
    else:
        print(f"  All {len(all_sc_cats)} categories represented in {split}")

if not category_balance_ok:
    raise RuntimeError("Speech Commands category balance check failed.")

print("Speech Commands category balance preserved with 3500 selected files.")

## 9. Dataset Manifest Creation

Build the full manifest with all required fields including SHA-256 hashes.

In [ ]:
# ============================================================
# SECTION 9 - BUILD FULL MANIFEST
# ============================================================

print("Building manifest with SHA-256 hashes...")
print()

manifest_rows = []

print("Hashing positive files...")
for rec in tqdm(positive_records, desc="Positive"):
    f = rec["filepath"]
    sr, ch, dur = get_wav_info(f)
    manifest_rows.append({
        "source_path": str(f),
        "relative_source_path": str(f.relative_to(PROJECT_ROOT)),
        "filename": f.name,
        "label": LABEL_POSITIVE,
        "category": "positive",
        "subcategory": "vaani",
        "speaker_or_source_id": rec["speaker"],
        "source_recording_id": rec["speaker"],
        "split": rec["split"],
        "sha256": rec["sha256"],
        "sample_rate": sr,
        "channels": ch,
        "duration": dur,
    })

print("Hashing silence files...")
for rec in tqdm(silence_records, desc="Silence"):
    f = rec["filepath"]
    sr, ch, dur = get_wav_info(f)
    manifest_rows.append({
        "source_path": str(f),
        "relative_source_path": str(f.relative_to(PROJECT_ROOT)),
        "filename": f.name,
        "label": LABEL_NEGATIVE,
        "category": "negative_silence",
        "subcategory": "silence",
        "speaker_or_source_id": rec["source_recording_id"],
        "source_recording_id": rec["source_recording_id"],
        "split": rec["split"],
        "sha256": file_sha256(f),
        "sample_rate": sr,
        "channels": ch,
        "duration": dur,
    })

print("Hashing background files...")
for rec in tqdm(bg_records, desc="Background"):
    f = rec["filepath"]
    sr, ch, dur = get_wav_info(f)
    manifest_rows.append({
        "source_path": str(f),
        "relative_source_path": str(f.relative_to(PROJECT_ROOT)),
        "filename": f.name,
        "label": LABEL_NEGATIVE,
        "category": "negative_background",
        "subcategory": rec["source_recording_id"],
        "speaker_or_source_id": rec["source_recording_id"],
        "source_recording_id": rec["source_recording_id"],
        "split": rec["split"],
        "sha256": file_sha256(f),
        "sample_rate": sr,
        "channels": ch,
        "duration": dur,
    })

print("Hashing speech command files...")
for rec in tqdm(sc_records, desc="Speech Cmds"):
    f = rec["filepath"]
    sr, ch, dur = get_wav_info(f)
    manifest_rows.append({
        "source_path": str(f),
        "relative_source_path": str(f.relative_to(PROJECT_ROOT)),
        "filename": f.name,
        "label": LABEL_NEGATIVE,
        "category": "negative_speech_commands",
        "subcategory": rec["subcategory"],
        "speaker_or_source_id": extract_sc_speaker(f),
        "source_recording_id": f.name,
        "split": rec["split"],
        "sha256": rec["sha256"],
        "sample_rate": sr,
        "channels": ch,
        "duration": dur,
    })

manifest_df = pd.DataFrame(manifest_rows)

print(f"\nManifest built: {len(manifest_df)} rows")
print(f"Columns: {list(manifest_df.columns)}")

if len(manifest_df) != EXPECTED_COUNTS["grand_total"]:
    raise RuntimeError(
        f"Manifest row count mismatch: {len(manifest_df)} != {EXPECTED_COUNTS['grand_total']}"
    )

if manifest_df["source_path"].duplicated().any():
    duplicate_paths = manifest_df[manifest_df["source_path"].duplicated(keep=False)]["source_path"].tolist()
    raise RuntimeError(f"Manifest contains duplicate source_path entries: {duplicate_paths[:10]}")

positive_manifest_df = manifest_df[manifest_df["category"] == "positive"]
positive_discovered = {str(p.resolve()) for p in positive_files}
positive_manifest_paths = [str(Path(p).resolve()) for p in positive_manifest_df["source_path"]]
positive_manifest_set = set(positive_manifest_paths)

if len(positive_manifest_paths) != len(positive_manifest_set):
    raise RuntimeError("Positive manifest entries are not unique by source_path.")

missing_positive = sorted(positive_discovered - positive_manifest_set)
extra_positive = sorted(positive_manifest_set - positive_discovered)
if missing_positive or extra_positive:
    print(f"Missing positive manifest entries: {missing_positive[:10]}")
    print(f"Unexpected positive manifest entries: {extra_positive[:10]}")
    raise RuntimeError("Positive manifest does not match discovered positive files exactly.")

print(
    "Verified every discovered positive file has exactly one unique manifest entry "
    f"({len(positive_manifest_paths)} entries)."
)

In [ ]:
# ============================================================
# SHA-256 DUPLICATE REPORT - NO ROW REMOVAL
# ============================================================
# Duplicate audio is allowed only when every identical SHA-256 group stays
# inside one split. Do not drop rows: all 704 positive files and all 3500
# selected Speech Commands files must remain in the manifest.

print("=" * 60)
print("SHA-256 DUPLICATE REPORT (NO ROW REMOVAL)")
print("=" * 60)

hash_counts = manifest_df["sha256"].value_counts()
dupe_hashes = list(hash_counts[hash_counts > 1].index)

same_split_duplicate_groups = 0
cross_split_duplicate_groups = []

for h in sorted(dupe_hashes):
    dupe_rows = manifest_df[manifest_df["sha256"] == h]
    splits_in_group = set(dupe_rows["split"])
    if len(splits_in_group) > 1:
        cross_split_duplicate_groups.append((h, dupe_rows))
    else:
        same_split_duplicate_groups += 1

print(f"  Duplicate SHA-256 groups:           {len(dupe_hashes)}")
print(f"  Same-split duplicate groups:        {same_split_duplicate_groups}")
print(f"  Cross-split duplicate groups:       {len(cross_split_duplicate_groups)}")
print(f"  Manifest rows preserved:            {len(manifest_df)}")

if cross_split_duplicate_groups:
    print()
    print("Cross-split duplicate audio examples:")
    for h, rows in cross_split_duplicate_groups[:10]:
        print(f"  Hash {h[:16]}...")
        for _, row in rows[["filename", "category", "split"]].iterrows():
            print(f"    {row['filename']} | {row['category']} | {row['split']}")
    raise RuntimeError("Identical audio crosses train/validation/test. Do not copy or train.")

if dupe_hashes:
    print("  Duplicate audio exists, but all duplicate groups are confined to one split.")
else:
    print("  No duplicate audio hashes found.")

## 10. Copy Dataset into model_v2/data/

Only copies files — never modifies `dataset/raw/`.

In [ ]:
# ============================================================
# SECTION 10 - COPY FILES INTO model_v2/data/
# ============================================================

print("=" * 60)
print("COPYING FILES TO model_v2/data/")
print("=" * 60)

print("Clearing previous copied WAVs from split/category folders...")
removed_existing = 0
for split in SPLITS:
    for cat in CATEGORIES:
        dst_dir = V2_DIRS["data"] / split / cat
        dst_dir.mkdir(parents=True, exist_ok=True)
        for old_wav in dst_dir.glob("*.wav"):
            old_wav.unlink()
            removed_existing += 1
print(f"  Removed old copied WAVs: {removed_existing}")

copy_count = 0
copy_errors = []

for _, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Copying"):
    src = Path(row["source_path"])
    dst_dir = V2_DIRS["data"] / row["split"] / row["category"]
    dst = dst_dir / row["filename"]

    try:
        shutil.copy2(str(src), str(dst))
        copy_count += 1
    except Exception as e:
        copy_errors.append(f"{src.name}: {e}")

print(f"\nCopied: {copy_count} files")
if copy_errors:
    print(f"Copy errors: {len(copy_errors)}")
    for err in copy_errors[:10]:
        print(f"  {err}")
    raise RuntimeError("Copy failed. Do not continue.")

print("No copy errors.")

print()
print("Verification - files in model_v2/data/:")
copy_counts_ok = True
for split in SPLITS:
    for cat in CATEGORIES:
        d = V2_DIRS["data"] / split / cat
        n = len(list(d.glob("*.wav")))
        expected = len(manifest_df[
            (manifest_df["split"] == split) & (manifest_df["category"] == cat)
        ])
        status = "OK" if n == expected else "FAIL"
        print(f"  {status:4s} {split:12s} / {cat:30s}: {n:>5d} (expected {expected})")
        if n != expected:
            copy_counts_ok = False

if not copy_counts_ok:
    raise RuntimeError("Copied file counts do not match manifest. Do not continue.")

In [ ]:
# ── Save manifest ────────────────────────────────────────────

csv_path = V2_DIRS["manifests"] / "v2_dataset_manifest.csv"
manifest_df.to_csv(csv_path, index=False)
print(f"Saved manifest CSV:  {csv_path}")
print(f"  Rows: {len(manifest_df)}")

json_path = V2_DIRS["manifests"] / "v2_dataset_manifest.json"
manifest_df.to_json(json_path, orient="records", indent=2)
print(f"Saved manifest JSON: {json_path}")

## 11. Leakage Verification

Multi-level leakage checks:
1. **Hash-based**: No identical SHA-256 hash across splits
2. **Filename-based**: No identical filename across splits within same category
3. **Source-recording-based**: No source recording ID appears in multiple splits
4. **Speaker-based (positive)**: Verify speaker coverage is intentional

In [ ]:
# ============================================================
# SECTION 11 - LEAKAGE VERIFICATION
# ============================================================

print("=" * 60)
print("LEAKAGE VERIFICATION")
print("=" * 60)

leakage_found = False

print("\nCheck 1: Global SHA-256 - no identical audio across splits")
for i, s1 in enumerate(SPLITS):
    for s2 in SPLITS[i + 1:]:
        hashes_s1 = set(manifest_df[manifest_df["split"] == s1]["sha256"])
        hashes_s2 = set(manifest_df[manifest_df["split"] == s2]["sha256"])
        overlap = hashes_s1 & hashes_s2
        if overlap:
            print(f"  FAIL {s1} vs {s2}: {len(overlap)} duplicate hashes cross splits")
            leakage_found = True
        else:
            print(f"  OK   {s1} vs {s2}: no hash overlap")

print("\nCheck 2: Filename uniqueness within each category")
for cat in CATEGORIES:
    cat_df = manifest_df[manifest_df["category"] == cat]
    for i, s1 in enumerate(SPLITS):
        for s2 in SPLITS[i + 1:]:
            fns1 = set(cat_df[cat_df["split"] == s1]["filename"])
            fns2 = set(cat_df[cat_df["split"] == s2]["filename"])
            overlap = fns1 & fns2
            if overlap:
                print(f"  FAIL {cat}: {len(overlap)} filenames in both {s1} and {s2}")
                leakage_found = True
            elif fns1 and fns2:
                print(f"  OK   {cat}: no filename overlap between {s1} and {s2}")

print("\nCheck 3: Silence/background source recordings stay in one split")
for cat in ["negative_silence", "negative_background"]:
    cat_df = manifest_df[manifest_df["category"] == cat]
    source_splits = cat_df.groupby("source_recording_id")["split"].apply(set)
    leaked_sources = {
        source_id: splits
        for source_id, splits in source_splits.items()
        if len(splits) > 1
    }
    if leaked_sources:
        leakage_found = True
        for source_id, splits in leaked_sources.items():
            print(f"  FAIL {cat}: source {source_id!r} appears in {splits}")
    print(f"  {'FAIL' if leaked_sources else 'OK  '} {cat}: {len(source_splits)} sources, {len(leaked_sources)} leaked")

print("\nCheck 4: Positive speaker coverage")
pos_df = manifest_df[manifest_df["category"] == "positive"]
for spk in sorted(pos_df["speaker_or_source_id"].unique()):
    spk_splits = set(pos_df[pos_df["speaker_or_source_id"] == spk]["split"])
    if spk_splits == set(SPLITS):
        print(f"  OK   {spk}: present in all 3 splits")
    else:
        print(f"  FAIL {spk}: only in {spk_splits}")
        leakage_found = True

print("\nCheck 5: Duplicate hashes are same-split only")
hash_counts = manifest_df["sha256"].value_counts()
dupe_hashes = hash_counts[hash_counts > 1]
same_split_dupes = 0
cross_split_dupes = 0
for h in dupe_hashes.index:
    splits = set(manifest_df[manifest_df["sha256"] == h]["split"])
    if len(splits) > 1:
        cross_split_dupes += 1
    else:
        same_split_dupes += 1
print(f"  Duplicate hash groups:     {len(dupe_hashes)}")
print(f"  Same-split duplicate groups: {same_split_dupes}")
print(f"  Cross-split duplicate groups: {cross_split_dupes}")
if cross_split_dupes:
    leakage_found = True

print()
if leakage_found:
    print("LEAKAGE DETECTED - review the issues above.")
    raise RuntimeError("Leakage verification failed. Do not train.")

print("NO LEAKAGE DETECTED - dataset splits are clean.")

## 12. Dataset Statistics

In [ ]:
# ============================================================
# SECTION 12 - DATASET STATISTICS
# ============================================================

print("=" * 70)
print("  V2 DATASET SUMMARY")
print("=" * 70)

print()
for cat in CATEGORIES:
    cat_df = manifest_df[manifest_df["category"] == cat]
    cat_label = {
        "positive": "Positive (Vaani)",
        "negative_silence": "Silence",
        "negative_background": "Background",
        "negative_speech_commands": "Speech Commands",
    }[cat]
    print(f"{cat_label}:")
    for split in SPLITS:
        n = len(cat_df[cat_df["split"] == split])
        print(f"  {split:12s}: {n:>5d}")
    print(f"  {'total':12s}: {len(cat_df):>5d}")
    print()

print("Total:")
for split in SPLITS:
    sdf = manifest_df[manifest_df["split"] == split]
    n_pos = len(sdf[sdf["label"] == LABEL_POSITIVE])
    n_neg = len(sdf[sdf["label"] == LABEL_NEGATIVE])
    print(f"  {split:12s}: {len(sdf):>5d}  (pos: {n_pos:>4d}, neg: {n_neg:>4d})")
print(f"  {'GRAND TOTAL':12s}: {len(manifest_df):>5d}")

print()
print("-" * 70)
print("MANDATORY COUNT VERIFICATION")
print("-" * 70)
category_totals = manifest_df.groupby("category").size().to_dict()
observed_counts = {
    "positive": int(category_totals.get("positive", 0)),
    "negative_silence": int(category_totals.get("negative_silence", 0)),
    "negative_background": int(category_totals.get("negative_background", 0)),
    "negative_speech_commands": int(category_totals.get("negative_speech_commands", 0)),
    "grand_total": int(len(manifest_df)),
}

summary_counts_ok = True
count_lines = [
    ("positive train + validation + test", "positive"),
    ("silence", "negative_silence"),
    ("background", "negative_background"),
    ("speech_commands", "negative_speech_commands"),
    ("grand total", "grand_total"),
]
for label, key in count_lines:
    actual = observed_counts[key]
    expected = EXPECTED_COUNTS[key]
    ok = actual == expected
    summary_counts_ok = summary_counts_ok and ok
    status = "OK" if ok else "FAIL"
    print(f"  {status:4s} {label} == {actual} (expected {expected})")

print()
print("-" * 70)
print("SPEAKERS")
print("-" * 70)
pos_df = manifest_df[manifest_df["category"] == "positive"]
print(f"Number of speakers: {pos_df['speaker_or_source_id'].nunique()}")
print()
for spk in sorted(pos_df["speaker_or_source_id"].unique()):
    spk_df = pos_df[pos_df["speaker_or_source_id"] == spk]
    spk_splits = {split: len(spk_df[spk_df["split"] == split]) for split in SPLITS}
    print(
        f"  {spk:<12s}: total={len(spk_df):>4d}  "
        f"train={spk_splits['train']:>3d}  "
        f"val={spk_splits['validation']:>3d}  "
        f"test={spk_splits['test']:>3d}"
    )

print()
print("-" * 70)
print("SPEECH COMMANDS CATEGORIES")
print("-" * 70)
sc_df = manifest_df[manifest_df["category"] == "negative_speech_commands"]
sc_cats_summary = sc_df.groupby("subcategory").size().sort_index()
print(f"Total categories: {len(sc_cats_summary)}")
for cat_name, cnt in sc_cats_summary.items():
    print(f"  {cat_name:<20s}: {cnt:>4d}")

print()
print("-" * 70)
print("SOURCE RECORDINGS PER SPLIT")
print("-" * 70)
for cat in ["negative_silence", "negative_background"]:
    cat_df = manifest_df[manifest_df["category"] == cat]
    cat_label = "Silence" if "silence" in cat else "Background"
    for split in SPLITS:
        sources = cat_df[cat_df["split"] == split]["source_recording_id"].unique()
        print(f"  {cat_label} {split:12s}: {len(sources)} source(s): {list(sources)}")

print()
print("-" * 70)
print("INVALID FILES")
print("-" * 70)
total_issues_found = sum(1 for r in all_val_results if r["issues"])
if total_issues_found == 0:
    print("  No invalid files found during validation.")
else:
    print(f"  {total_issues_found} files had validation issues.")

print()
print("-" * 70)
print("CLASS BALANCE")
print("-" * 70)
n_pos_total = len(manifest_df[manifest_df["label"] == LABEL_POSITIVE])
n_neg_total = len(manifest_df[manifest_df["label"] == LABEL_NEGATIVE])
ratio = n_neg_total / n_pos_total if n_pos_total > 0 else float("inf")
print(f"  Total positive: {n_pos_total}")
print(f"  Total negative: {n_neg_total}")
print(f"  Ratio positive:negative = 1:{ratio:.2f}")
print("  Intentionally negative-heavy; class imbalance is handled during training.")

print()
print(f"  Random seed: {SEED}")

## 13. Final Sanity Checks

In [ ]:
# ============================================================
# SECTION 13 - FINAL SANITY CHECKS
# ============================================================

print("=" * 60)
print("SANITY CHECKS")
print("=" * 60)

checks_passed = 0
checks_total = 0


def record_check(name, ok, detail=""):
    global checks_passed, checks_total
    checks_total += 1
    if ok:
        checks_passed += 1
        print(f"OK   {name}{': ' + detail if detail else ''}")
    else:
        print(f"FAIL {name}{': ' + detail if detail else ''}")


category_counts = manifest_df.groupby("category").size().to_dict()
mandatory_counts_ok = (
    category_counts.get("positive", 0) == EXPECTED_COUNTS["positive"]
    and category_counts.get("negative_silence", 0) == EXPECTED_COUNTS["negative_silence"]
    and category_counts.get("negative_background", 0) == EXPECTED_COUNTS["negative_background"]
    and category_counts.get("negative_speech_commands", 0) == EXPECTED_COUNTS["negative_speech_commands"]
    and len(manifest_df) == EXPECTED_COUNTS["grand_total"]
)
record_check("Mandatory category and grand-total counts", mandatory_counts_ok)

positive_discovered = {str(p.resolve()) for p in positive_files}
pos_df_check = manifest_df[manifest_df["category"] == "positive"]
positive_manifest_paths = [str(Path(p).resolve()) for p in pos_df_check["source_path"]]
positive_manifest_set = set(positive_manifest_paths)
unique_positive_manifest_ok = (
    len(positive_manifest_paths) == len(positive_manifest_set)
    and positive_manifest_set == positive_discovered
    and len(positive_manifest_paths) == EXPECTED_COUNTS["positive"]
)
record_check("Every discovered positive file has one unique manifest entry", unique_positive_manifest_ok)

all_source_paths = manifest_df["source_path"].tolist()
record_check("Manifest source_path values are globally unique", len(all_source_paths) == len(set(all_source_paths)))

filename_leakage_ok = True
for cat in CATEGORIES:
    cat_df = manifest_df[manifest_df["category"] == cat]
    for i, s1 in enumerate(SPLITS):
        for s2 in SPLITS[i + 1:]:
            fns1 = set(cat_df[cat_df["split"] == s1]["filename"])
            fns2 = set(cat_df[cat_df["split"] == s2]["filename"])
            if fns1 & fns2:
                filename_leakage_ok = False
                print(f"  Filename overlap in {cat}: {s1} vs {s2}: {sorted(fns1 & fns2)[:10]}")
record_check("No filename leakage across splits", filename_leakage_ok)

unique_labels = set(manifest_df["label"].unique())
record_check("Labels are binary {0, 1}", unique_labels.issubset({0, 1}))

all_speakers_in_all = True
for spk in sorted(pos_df_check["speaker_or_source_id"].unique()):
    spk_splits = set(pos_df_check[pos_df_check["speaker_or_source_id"] == spk]["split"])
    if spk_splits != set(SPLITS):
        all_speakers_in_all = False
        print(f"  Speaker {spk} only in {spk_splits}")
record_check("Every positive speaker appears in all 3 splits", all_speakers_in_all)

source_ok = True
for cat in ["negative_silence", "negative_background"]:
    cat_df = manifest_df[manifest_df["category"] == cat]
    for src_id in cat_df["source_recording_id"].unique():
        src_splits = set(cat_df[cat_df["source_recording_id"] == src_id]["split"])
        if len(src_splits) > 1:
            source_ok = False
            print(f"  {cat} source {src_id!r} appears in multiple splits: {src_splits}")
record_check("Silence/background source-aware splitting preserved", source_ok)

silence_train_only_ok = (
    len(manifest_df[(manifest_df["category"] == "negative_silence") & (manifest_df["split"] == "train")])
    == EXPECTED_COUNTS["negative_silence"]
    and len(manifest_df[(manifest_df["category"] == "negative_silence") & (manifest_df["split"] != "train")]) == 0
)
record_check("All 307 silence clips are in train", silence_train_only_ok)

sc_df_final = manifest_df[manifest_df["category"] == "negative_speech_commands"]
all_cats = set(sc_df_final["subcategory"].unique())
sc_repr_ok = len(all_cats) == EXPECTED_SC_CATEGORIES and len(sc_df_final) == EXPECTED_COUNTS["negative_speech_commands"]
for cat in sorted(all_cats):
    cat_count = len(sc_df_final[sc_df_final["subcategory"] == cat])
    if cat_count != TARGET_PER_SC_CATEGORY:
        sc_repr_ok = False
        print(f"  Speech Commands category {cat!r}: {cat_count} files")
for split in SPLITS:
    split_cats = set(sc_df_final[sc_df_final["split"] == split]["subcategory"].unique())
    if split_cats != all_cats:
        sc_repr_ok = False
        print(f"  {split} missing Speech Commands categories: {sorted(all_cats - split_cats)}")
record_check("3500 Speech Commands with category balance", sc_repr_ok)

files_ok = True
for split in SPLITS:
    for cat in CATEGORIES:
        d = V2_DIRS["data"] / split / cat
        n_actual = len(list(d.glob("*.wav")))
        n_expected = len(manifest_df[
            (manifest_df["split"] == split) & (manifest_df["category"] == cat)
        ])
        if n_actual != n_expected:
            files_ok = False
            print(f"  {split}/{cat}: {n_actual} copied files (expected {n_expected})")
record_check("Copied file counts match manifest", files_ok)

hash_dupe_across = False
for i, s1 in enumerate(SPLITS):
    for s2 in SPLITS[i + 1:]:
        h1 = set(manifest_df[manifest_df["split"] == s1]["sha256"])
        h2 = set(manifest_df[manifest_df["split"] == s2]["sha256"])
        overlap = h1 & h2
        if overlap:
            hash_dupe_across = True
            print(f"  {len(overlap)} duplicate SHA-256 hashes between {s1} and {s2}")
record_check("No SHA-256 duplicate audio crosses train/validation/test", not hash_dupe_across)

manifest_paths = {str(Path(p).resolve()) for p in manifest_df["source_path"]}
validated_paths = {str(Path(r["filepath"]).resolve()) for r in all_val_results}
validation_issue_count = sum(1 for r in all_val_results if r["issues"])
validation_ok = (
    validation_issue_count == 0
    and manifest_paths.issubset(validated_paths)
    and len(validated_paths) == EXPECTED_COUNTS["grand_total"]
)
record_check("All selected files were validated", validation_ok)

summary_ok = bool(summary_counts_ok)
record_check("Dataset summary mandatory counts passed", summary_ok)

print(f"\n{'=' * 60}")
print(f"SANITY CHECKS: {checks_passed}/{checks_total} PASSED")
print(f"{'=' * 60}")

dataset_ready = checks_passed == checks_total
if not dataset_ready:
    raise RuntimeError(
        "Mandatory dataset checks failed. Do not run augmentation, feature extraction, or training."
    )

print("All mandatory checks passed.")

In [ ]:
# ============================================================
# FINAL MESSAGE
# ============================================================

if "dataset_ready" not in globals() or not dataset_ready:
    raise RuntimeError("Dataset is not ready. Do not continue to Notebook 02.")

print()
print("=" * 70)
print("  V2 DATASET PREPARATION COMPLETE")
print("=" * 70)
print()
print("Outputs:")
print(f"  Data:      {V2_DIRS['data']}")
print(f"  Manifest:  {csv_path}")
print(f"  Manifest:  {json_path}")
print()
print("Mandatory counts:")
print(f"  positive train + validation + test == {EXPECTED_COUNTS['positive']}")
print(f"  silence == {EXPECTED_COUNTS['negative_silence']}")
print(f"  background == {EXPECTED_COUNTS['negative_background']}")
print(f"  speech_commands == {EXPECTED_COUNTS['negative_speech_commands']}")
print(f"  grand total == {EXPECTED_COUNTS['grand_total']}")
print()
print("Next step is Notebook 02 only after this successful preparation run.")
print("DO NOT modify anything under dataset/raw/.")
print("DO NOT train a model in this notebook.")
print("=" * 70)
print("DATASET READY")